# Spectral-MAE: corrected band-masking study

**Why this rerun:** in the first band study every training image had 3 of 10 bands hidden, so the encoder never saw a complete image during pre-training but always saw complete images at evaluation (train/test mismatch). Fix: bands are hidden in only **50% of training images** (`band_mask_prob = 0.5`); the rest are complete.

**Declared before running.** 10 bands (B2–B8, B8A, B11, B12), 75% spatial masking, summed MSE, 50 epochs, **fresh seeds 3–5**.

| Arm | Band masking (3 of 10 bands, in 50% of images) | NDVI loss |
|---|---|---|
| A | off | none |
| B | on | none |
| C | on | brightness-weighted, λ = 10 |

* **Primary comparison: C vs B.** Secondary: B vs A.
* A gain counts only if it is positive on all 3 seeds, for both the [CLS] and the mean-pooled probe.

**Before running:** upload the newest `spectral-mae.zip` (it has `band_mask_prob`) as a new input and remove older ones; GPU + Internet on; **Save Version → Save & Run All (Commit)**. About 2 hours.

In [1]:
# 1. Copy the code (needs the version with band masking)
import glob, os, shutil, subprocess
cands = glob.glob('/kaggle/input/**/configs/bands10.yaml', recursive=True)
cands = [x for x in cands if 'band_mask_prob' in open(x.replace('bands10.yaml', 'default.yaml')).read()]
if not cands:
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/kaggle/tmp_code'], check=True)
    cands = glob.glob('/kaggle/tmp_code/**/configs/bands10.yaml', recursive=True)
    cands = [x for x in cands if 'band_mask_prob' in open(x.replace('bands10.yaml', 'default.yaml')).read()]
assert cands, 'Upload the NEWEST spectral-mae.zip (the one with band_mask_prob)'
repo = os.path.dirname(os.path.dirname(cands[0]))
shutil.copytree(repo, '/kaggle/working/spectral-mae', dirs_exist_ok=True)
os.chdir('/kaggle/working/spectral-mae')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['EPOCHS'] = '50'
print('Code at', os.getcwd())

Code at /kaggle/working/spectral-mae


In [2]:
# 2. Dependencies, GPU check, unit tests (must say '23 passed')
!pip install -q datasets tifffile
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python -m pytest -q

CUDA: True Tesla T4
.......................                                                  [100%]
23 passed in 9.95s


In [3]:
# 3. Data: 10-band cache (check 'Band order looks correct.' and 'train 18898, val 2700, test 5400')
!python -m spectral_mae.prep_data --from-hf --config configs/bands10.yaml

README.md: 100%|███████████████████████████| 4.56k/4.56k [00:00<00:00, 2.53MB/s]

data/train-00000-of-00005.parquet: downloading bytes: ███▉  |  204MB, 18.0MB/s  
data/train-00000-of-00005.parquet: downloading bytes: █████▍|  285MB, 24.7MB/s  
data/train-00000-of-00005.parquet: reconstructing file:  86%|▊|  268MB /  314MB,
data/train-00000-of-00005.parquet: downloading bytes: ██████|  305MB, 27.0MB/s  
data/train-00000-of-00005.parquet: reconstructing file: 100%|█|  314MB /  314MB,

data/train-00001-of-00005.parquet: downloading bytes: █████▌|  294MB, 24.7MB/s  
data/train-00001-of-00005.parquet: downloading bytes: ██████|  311MB, 28.8MB/s  
data/train-00001-of-00005.parquet: reconstructing file: 100%|█|  319MB /  319MB,

data/train-00002-of-00005.parquet: downloading bytes: ████▉ |  254MB, 21.9MB/s  
data/train-00002-of-00005.parquet: downloading bytes: █████▌|  290MB, 26.1MB/s  
data/train-00002-of-00005.parquet: downloading bytes: ██████|  303MB, 26.9MB/s  
data/train-00002-of-00005

In [4]:
%%bash
# 4. Gate: 1-epoch run of arm C must finish with finite losses before the real runs start
set -e
python -m spectral_mae.pretrain --config configs/bands10.yaml --name gate --set pretrain.epochs=1 \
  --set pretrain.warmup_epochs=1 --set loss.lambda_ndvi=10 --set loss.ndvi_weighting=brightness \
  --set loss.mse_reduction=sum_per_patch --set model.band_mask_prob=0.5 --no-viz
grep -q '"bad_steps": 0' runs/gate_*/summary.json && echo "GATE PASSED"
rm -rf runs/gate_*

Device cuda | params {'total': 3654208, 'encoder': 3161472, 'decoder': 492736} | encoder MACs/img masked 13,328,640 vs full 45,787,392
epoch   1/1 | loss 149.1765 mse 143.1447 ndvi 0.6032 | val mse 0.0079 ndvi_mae 0.3095 | 21.5s
Done in 0.4 min. Checkpoint: runs/gate_20261116-073842/last.pt
GATE PASSED


In [5]:
%%bash
# 5. Pre-training: arms A, B, C x fresh seeds 3-5 (band masking in 50% of images)
C="--config configs/bands10.yaml --set loss.mse_reduction=sum_per_patch --set pretrain.epochs=$EPOCHS --set model.band_mask_prob=0.5 --no-viz"
for s in 3 4 5; do
  python -m spectral_mae.pretrain $C --name b10fix_A_s$s --set model.band_mask_ratio=0 --set loss.lambda_ndvi=0 --set seed=$s
  python -m spectral_mae.pretrain $C --name b10fix_B_s$s --set loss.lambda_ndvi=0 --set seed=$s
  python -m spectral_mae.pretrain $C --name b10fix_C_s$s --set loss.lambda_ndvi=10 --set loss.ndvi_weighting=brightness --set seed=$s
done

Device cuda | params {'total': 3652288, 'encoder': 3161472, 'decoder': 490816} | encoder MACs/img masked 13,328,640 vs full 45,787,392
epoch   1/50 | loss 214.5072 mse 214.5072 ndvi 0.7363 | val mse 0.0542 ndvi_mae 0.7051 | 14.4s
epoch   2/50 | loss 71.5134 mse 71.5134 ndvi 0.5540 | val mse 0.0100 ndvi_mae 0.3870 | 10.6s
epoch   3/50 | loss 13.7983 mse 13.7983 ndvi 0.3020 | val mse 0.0033 ndvi_mae 0.2589 | 11.0s
epoch   4/50 | loss 6.9419 mse 6.9419 ndvi 0.2209 | val mse 0.0025 ndvi_mae 0.1899 | 10.8s
epoch   5/50 | loss 5.6912 mse 5.6912 ndvi 0.1647 | val mse 0.0021 ndvi_mae 0.1428 | 10.7s
epoch   6/50 | loss 5.2684 mse 5.2684 ndvi 0.1447 | val mse 0.0021 ndvi_mae 0.1398 | 10.6s
epoch   7/50 | loss 5.1963 mse 5.1963 ndvi 0.1422 | val mse 0.0020 ndvi_mae 0.1345 | 10.6s
epoch   8/50 | loss 5.1363 mse 5.1363 ndvi 0.1415 | val mse 0.0021 ndvi_mae 0.1437 | 10.6s
epoch   9/50 | loss 5.0991 mse 5.0991 ndvi 0.1420 | val mse 0.0020 ndvi_mae 0.1354 | 10.5s
epoch  10/50 | loss 4.9987 mse 4.9987 

In [6]:
%%bash
# 6. Probes: [CLS] and mean-pooled, 3 probe seeds each
for ck in runs/b10fix_*/last.pt; do
  python -m spectral_mae.probe --config configs/bands10.yaml --ckpt "$ck"
  python -m spectral_mae.probe --config configs/bands10.yaml --ckpt "$ck" --set probe.pool=mean
done

Extracting cls features on cuda ...
  labels  10.0% ( 1890) seed 0: acc 68.26  macro-F1 66.83
  labels  50.0% ( 9449) seed 0: acc 74.76  macro-F1 73.80
  labels 100.0% (18898) seed 0: acc 76.65  macro-F1 75.80
  labels  10.0% ( 1890) seed 1: acc 68.98  macro-F1 67.79
  labels  50.0% ( 9449) seed 1: acc 74.52  macro-F1 73.60
  labels 100.0% (18898) seed 1: acc 77.00  macro-F1 76.20
  labels  10.0% ( 1890) seed 2: acc 69.44  macro-F1 68.11
  labels  50.0% ( 9449) seed 2: acc 74.59  macro-F1 73.51
  labels 100.0% (18898) seed 2: acc 76.93  macro-F1 76.08

### Linear probe: probe_b10fix_A_s3_20261116-073910

| Labels | Seeds | top1_acc | macro_f1 |
|---|---|---|---|
| 10% | 3 | 68.90 ± 0.60 | 67.58 ± 0.67 |
| 50% | 3 | 74.62 ± 0.12 | 73.64 ± 0.15 |
| 100% | 3 | 76.86 ± 0.19 | 76.02 ± 0.21 |

Saved runs/probe_b10fix_A_s3_20261116-073910_20261116-090316/results.md
Extracting mean features on cuda ...
  labels  10.0% ( 1890) seed 0: acc 68.28  macro-F1 66.84
  labels  50.0% ( 9449) seed 0: ac

In [7]:
%%bash
# 7. Per-class gradient diagnostics (seed 3, arms B and C)
for ck in runs/b10fix_B_s3_*/last.pt runs/b10fix_C_s3_*/last.pt; do python -m spectral_mae.diagnose_gradients --ckpt "$ck"; done

| Class | n | mean NIR+Red | NDVI err | grad MSE | grad NDVI | NDVI/MSE |
|---|---|---|---|---|---|---|
| AnnualCrop | 300 | 0.450 | 0.140 | 0.000518 | 0.0642 | 123.88 |
| Forest | 300 | 0.305 | 0.042 | 0.000261 | 0.108 | 415.17 |
| HerbaceousVegetation | 300 | 0.341 | 0.061 | 0.000295 | 0.085 | 288.10 |
| Highway | 250 | 0.342 | 0.150 | 0.000507 | 0.087 | 171.59 |
| Industrial | 250 | 0.347 | 0.164 | 0.000613 | 0.078 | 127.36 |
| Pasture | 200 | 0.409 | 0.085 | 0.000449 | 0.0755 | 168.01 |
| PermanentCrop | 250 | 0.410 | 0.093 | 0.000367 | 0.0683 | 185.97 |
| Residential | 300 | 0.330 | 0.123 | 0.000351 | 0.0837 | 238.33 |
| River | 250 | 0.273 | 0.270 | 0.000795 | 0.113 | 141.51 |
| SeaLake | 300 | 0.085 | 0.130 | 0.000146 | 0.439 | 3001.23 |

Saved runs/diag_b10fix_B_s3_20261116-074816_20261116-091806/per_class_gradients.png
| Class | n | mean NIR+Red | NDVI err | grad MSE | grad NDVI | NDVI/MSE |
|---|---|---|---|---|---|---|
| AnnualCrop | 300 | 0.450 | 0.138 | 0.000536 | 0.0283 |

In [8]:
# 8. Results
import json, glob, re
import numpy as np
from collections import defaultdict

def ms(v, d=2):
    v = np.asarray(v, dtype=float)
    return f"{v.mean():.{d}f} ± {v.std(ddof=1) if len(v) > 1 else 0:.{d}f}"

def probe_by_seed(key, prefix):
    acc = defaultdict(dict)
    for f in glob.glob(f'runs/{prefix}_{key}_s*/results.json'):
        s = int(re.search(rf'{prefix}_{key}_s(\d+)_', f).group(1))
        res = json.load(open(f))
        for frac in (0.1, 0.5, 1.0):
            acc[frac][s] = 100 * np.mean([r['top1_acc'] for r in res if r['frac'] == frac])
    return acc

arms = [('b10fix_A', 'A: no band mask, no NDVI'), ('b10fix_B', 'B: band mask, no NDVI'),
        ('b10fix_C', 'C: band mask + NDVI (bright, λ=10)')]
for prefix, title in [('probe', '[CLS] token probe'), ('meanprobe', 'Mean-pooled token probe')]:
    print(f"\n### {title}\n")
    print("| Arm | Val MSE (×1e-3) | Val NDVI err | Acc 10% | Acc 50% | Acc 100% |")
    print("|---|---|---|---|---|---|")
    per = {}
    for key, label in arms:
        summ = [json.load(open(f)) for f in sorted(glob.glob(f'runs/{key}_s*/summary.json'))]
        per[key] = probe_by_seed(key, prefix)
        print(f"| {label} | {ms([s['final_val_masked_mse'] * 1e3 for s in summ])} | "
              f"{ms([s['final_val_ndvi_mae'] for s in summ], 4)} | "
              + " | ".join(ms(list(per[key][f].values())) for f in (0.1, 0.5, 1.0)) + " |")
    for a, b, name in [('b10fix_C', 'b10fix_B', 'PRIMARY  C − B'), ('b10fix_B', 'b10fix_A', 'secondary B − A')]:
        print(f"\n{name}, per pre-training seed:")
        for f in (0.1, 0.5, 1.0):
            common = sorted(set(per[a][f]) & set(per[b][f]))
            d = [per[a][f][s] - per[b][f][s] for s in common]
            verdict = 'positive on all seeds' if d and all(x > 0 for x in d) else 'not consistent'
            print(f"  {int(f * 100):3d}% labels: {[round(float(x), 2) for x in d]}  mean {np.mean(d):+.2f}  ({verdict})")
print("\nVal MSE / NDVI err are measured with spatial masking only, so all arms are comparable.")
for f in sorted(glob.glob('runs/diag_b10fix_*/per_class_gradients.md')):
    print('\n==', f); print(open(f).read())


### [CLS] token probe

| Arm | Val MSE (×1e-3) | Val NDVI err | Acc 10% | Acc 50% | Acc 100% |
|---|---|---|---|---|---|
| A: no band mask, no NDVI | 1.71 ± 0.05 | 0.1227 ± 0.0021 | 68.84 ± 1.09 | 74.80 ± 0.93 | 76.84 ± 0.93 |
| B: band mask, no NDVI | 1.76 ± 0.01 | 0.1248 ± 0.0017 | 66.46 ± 0.76 | 73.17 ± 0.55 | 76.05 ± 0.38 |
| C: band mask + NDVI (bright, λ=10) | 1.82 ± 0.01 | 0.1191 ± 0.0011 | 67.19 ± 0.79 | 73.93 ± 0.60 | 76.64 ± 0.64 |

PRIMARY  C − B, per pre-training seed:
   10% labels: [0.91, 0.17, 1.09]  mean +0.73  (positive on all seeds)
   50% labels: [1.15, 0.06, 1.07]  mean +0.76  (positive on all seeds)
  100% labels: [0.75, -0.31, 1.33]  mean +0.59  (not consistent)

secondary B − A, per pre-training seed:
   10% labels: [-3.31, -3.07, -0.76]  mean -2.38  (not consistent)
   50% labels: [-2.09, -2.38, -0.43]  mean -1.63  (not consistent)
  100% labels: [-1.17, -1.33, 0.14]  mean -0.79  (not consistent)

### Mean-pooled token probe

| Arm | Val MSE (×1e-3) | Val NDVI 

In [9]:
# 9. Pack results (Output tab -> bandsfix_runs.zip)
!zip -qr /kaggle/working/bandsfix_runs.zip runs
print('Saved /kaggle/working/bandsfix_runs.zip')

Saved /kaggle/working/bandsfix_runs.zip
